# Logistic regression from scratch with gradient descent

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
warnings.simplefilter("ignore")

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Data that overlaps a little (so a best line exists)

In [ ]:
X, y = make_classification(n_samples=100, n_features=2, n_informative=2, n_redundant=0, n_classes=2,
                           n_clusters_per_class=1, random_state=4, class_sep=1.5)
sk = LogisticRegression(penalty=None, max_iter=100000, tol=1e-10).fit(X, y)
print("scikit-learn: intercept", sk.intercept_.round(4), "coef", sk.coef_.round(4), "accuracy", sk.score(X, y))

## Batch gradient descent: w = w + lr * X^T (y - y_hat) / m

In [ ]:
def gd(X, y, lr=0.5, epochs=5000):
    X = np.insert(X, 0, 1, axis=1)
    w = np.ones(X.shape[1])
    for _ in range(epochs):
        y_hat = sigmoid(X @ w)
        w = w + lr * (X.T @ (y - y_hat)) / X.shape[0]
    return w[1:], w[0]

for epochs in [100, 1000, 5000]:
    coef_, intercept_ = gd(X, y, epochs=epochs)
    p = sigmoid(X @ coef_ + intercept_)
    print(f"{epochs:5d} epochs: intercept {intercept_:.4f}, coef {coef_.round(4)}, log loss {log_loss(y, p):.5f}")
print("scikit-learn log loss", round(log_loss(y, sk.predict_proba(X)[:, 1]), 5))

## Perfectly separable data: the weights never stop growing

In [ ]:
Xs, ys = make_classification(n_samples=100, n_features=2, n_informative=1, n_redundant=0, n_classes=2,
                             n_clusters_per_class=1, random_state=41, hypercube=False, class_sep=20)
for epochs in [500, 5000, 50000]:
    coef_, intercept_ = gd(Xs, ys, epochs=epochs)
    p = sigmoid(Xs @ coef_ + intercept_)
    print(f"{epochs:6d} epochs: weights {np.r_[intercept_, coef_].round(2)}, log loss {log_loss(ys, p):.6f}")